In [21]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

print("🚀 Starting Data Processing Workflow...")

# =========================================================================
# 1. LOAD THE REAL FILE PIPELINES
# =========================================================================
try:
    df_customers = pd.read_csv('customers.csv', encoding='latin1')
    df_orders = pd.read_csv('orders.csv', encoding='latin1')
    df_items = pd.read_csv('order_items.csv', encoding='latin1')
    df_payments = pd.read_csv('payments.csv', encoding='latin1')
    df_products = pd.read_csv('products.csv', encoding='latin1')
    df_sellers = pd.read_csv('sellers.csv', encoding='latin1')
    print("✅ All 6 dataset files loaded successfully!")
except FileNotFoundError as e:
    print(f"❌ Error: Missing file in directory. Detail: {e}")
    print("Ensure all CSV filenames match Olist specifications and sit in this workspace.")
    raise

# Clean dates and filter out invalid/unfulfilled order entries upfront
df_orders['order_purchase_timestamp'] = pd.to_datetime(df_orders['order_purchase_timestamp'])
df_clean_orders = df_orders[~df_orders['order_status'].isin(['canceled', 'unavailable'])].copy()


#BASIC PROBLEMS:
# 1. List all unique cities where customers are located.
unique_cities = df_customers['customer_city'].unique()
print(unique_cities)
print("Total Rows:",len(unique_cities))

# 2. Count the number of orders placed in 2017

orders_2017 = df_orders[df_orders['order_purchase_timestamp'].dt.year == 2017]['order_id'].count()
print("Total Orders in 2017:", orders_2017)

#3. Find the total sales per category.

merged = pd.merge(df_items, df_products, on='product_id')
sales_per_cat = merged.groupby('product_category')['price'].sum().round(2).reset_index()
sales_per_cat = sales_per_cat.sort_values(by='price', ascending=False)
print(sales_per_cat)

#4. Calculate the percentage of orders that were paid in installments.

installment_orders = df_payments[df_payments['payment_installments'] > 1]['order_id'].nunique()
total_orders = df_payments['order_id'].nunique()
installment_pct = (installment_orders / total_orders) * 100

print(f"Installment Orders: {installment_orders}")
print(f"Total Orders: {total_orders}")
print(f"Installment Percentage: {installment_pct:.2f}%")

#5. Count the number of customers from each state.


state_counts = df_customers['customer_state'].value_counts().reset_index()
state_counts.columns = ['customer_state', 'customer_count']
print(state_counts)

#INTERMEDIATE PROBLEMS
#1. Calculate the number of orders per month in 2018

orders_2018 = df_orders[df_orders['order_purchase_timestamp'].dt.year == 2018].copy()
orders_2018['month'] = orders_2018['order_purchase_timestamp'].dt.month
monthly_orders = orders_2018.groupby('month')['order_id'].count().reset_index()
print(monthly_orders)

#2. Find the average number of products per order, grouped by customer city.

item_counts = df_items.groupby('order_id')['order_item_id'].count().reset_index()
merged_city = df_orders.merge(df_customers, on='customer_id').merge(item_counts, on='order_id')

avg_prod_city = merged_city.groupby('customer_city').agg(
    average_prod_per_order=('order_item_id', 'mean'),
    total_orders_placed=('order_id', 'count')
).reset_index()

avg_prod_city['average_prod_per_order'] = avg_prod_city['average_prod_per_order'].round(2)
avg_prod_city = avg_prod_city.sort_values(by='average_prod_per_order', ascending=False)

print(avg_prod_city.to_string(index=False))

#3. Calculate the percentage of total revenue contributed by each product category
merged = df_items.merge(df_products, on='product_id')
total_rev = df_items['price'].sum()
cat_rev = (merged.groupby('product_category')['price'].sum() / total_rev * 100).round(2).reset_index()
print(cat_rev.sort_values(by='price', ascending=False))

 #4. Identify the correlation between product price and the number of times a product has been purchased.
prod_stats = df_items.groupby('product_id').agg(
    purchase_count=('order_item_id', 'count'),
    avg_price=('price', 'mean')
)
corr = prod_stats['purchase_count'].corr(prod_stats['avg_price'])
print(f"Correlation: {corr:.4f}")

#5. Calculate the total revenue generated by each seller, and rank them by revenue.
seller_rev = df_items.groupby('seller_id')['price'].sum().round(2).reset_index()
seller_rev['revenue_rank'] = seller_rev['price'].rank(ascending=False, method='min').astype(int)
print(seller_rev.sort_values('revenue_rank'))

#ADVANCED PROBLEMS
#1. Calculate the moving average of order values for each customer over their order history.
cust_orders = df_customers.merge(df_orders, on='customer_id').merge(df_payments, on='order_id')
order_vals = cust_orders.groupby(['customer_unique_id', 'order_id', 'order_purchase_timestamp'])['payment_value'].sum().reset_index()
order_vals = order_vals.sort_values(['customer_unique_id', 'order_purchase_timestamp'])
order_vals['moving_avg'] = order_vals.groupby('customer_unique_id')['payment_value'].expanding().mean().round(2).reset_index(level=0, drop=True)
print(order_vals.head())


#2. Calculate the cumulative sales per month for each year.
sales_df = df_orders.merge(df_payments, on='order_id')
sales_df['year'] = sales_df['order_purchase_timestamp'].dt.year
sales_df['month'] = sales_df['order_purchase_timestamp'].dt.month
monthly = sales_df.groupby(['year', 'month'])['payment_value'].sum().reset_index()
monthly['cumulative_sales'] = monthly.groupby('year')['payment_value'].cumsum().round(2)
print(monthly.head(15))

#3. Calculate the year-over-year growth rate of total sales.
yearly = sales_df.groupby('year')['payment_value'].sum().reset_index()
yearly['yoy_growth_pct'] = (yearly['payment_value'].pct_change() * 100).round(2)
print(yearly)

#4. Calculate the retention rate of customers, defined as the percentage of customers who make another purchase
#within 6 months of their first purchase

df_orders['order_purchase_timestamp'] = pd.to_datetime(df_orders['order_purchase_timestamp'])
cust_ord = df_customers.merge(df_orders, on='customer_id')

# Sort by timestamp and order_id to pick each customer's first order
cust_ord = cust_ord.sort_values(['customer_unique_id', 'order_purchase_timestamp', 'order_id'])
first_orders = cust_ord.groupby('customer_unique_id').first().reset_index()[
    ['customer_unique_id', 'order_id', 'order_purchase_timestamp']
]
first_orders.columns = ['customer_unique_id', 'first_order_id', 'first_purchase']

# Merge back and count customers with a different order_id within 6 months
merged_ret = cust_ord.merge(first_orders, on='customer_unique_id')
retained = merged_ret[
    (merged_ret['order_id'] != merged_ret['first_order_id']) &
    (merged_ret['order_purchase_timestamp'] >= merged_ret['first_purchase']) &
    (merged_ret['order_purchase_timestamp'] <= merged_ret['first_purchase'] + pd.DateOffset(months=6))
]['customer_unique_id'].nunique()

total_cust = first_orders['customer_unique_id'].nunique()
retention_pct = (retained / total_cust) * 100

print(f"Retained Customers: {retained}")
print(f"Total Customers: {total_cust}")
print(f"Retention Rate: {retention_pct:.2f}%")

#5. Identify the top 3 customers who spent the most money in each year.

# Merge customers, orders, and order_items (matching your SQL query)
full_df = df_customers.merge(df_orders, on='customer_id').merge(df_items, on='order_id')
full_df['year'] = pd.to_datetime(full_df['order_purchase_timestamp']).dt.year

# Group by year and customer_unique_id to get total spent and order count
yearly_cust = full_df.groupby(['year', 'customer_unique_id']).agg(
    total_spent=('price', 'sum'),
    total_orders=('order_id', 'nunique')
).reset_index()

yearly_cust['total_spent'] = yearly_cust['total_spent'].round(2)

# Rank customers within each year using dense_rank
yearly_cust['rank'] = yearly_cust.groupby('year')['total_spent'].rank(method='dense', ascending=False).astype(int)

# Filter top 3 ranks and order columns to match your SQL output
top_3 = yearly_cust[yearly_cust['rank'] <= 3][['year', 'rank', 'customer_unique_id', 'total_spent', 'total_orders']]
top_3 = top_3.sort_values(['rank', 'year', 'customer_unique_id'])

print(top_3.to_string(index=False))





 year  rank               customer_unique_id  total_spent  total_orders
 2016     1 fdaa290acb9eeacb66fa7f979baa6803      1399.00             1
 2017     1 0a0a92112bd4c708ca5fde585afaa872     13440.00             1
 2018     1 763c8b1c9c68a0229c42c9fc6f662b93      7160.00             1
 2016     2 753bc5d6efa9e49a03e34cf521a9e124      1299.99             1
 2017     2 da122df9eeddfedc1dc1f5349a1a690c      7388.00             2
 2018     2 459bef486812aa25204be022145caa62      6729.00             1
 2016     3 b92a2e5e8a6eabcc80882c7d68b2c70b      1199.00             1
 2017     3 dc4802a71eae9be1dd28f5d788ceb526      6735.00             1
 2018     3 5d0a2980b292d049061542014e8960bf      4599.90             1
